[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/07_logistic_regression/08_linear_classifiers_cifar10.ipynb)

# Bài 08 — Bộ phân loại tuyến tính trên ảnh thật: CIFAR-10

**🎯 Mục tiêu**
- Làm việc với một bộ ảnh **thật**: CIFAR-10 — 60 000 ảnh màu 32×32, 10 lớp.
- Một mô hình tuyến tính $\mathbf s = W^\top \mathbf x$, **ba hàm mất mát**: One-vs-All logistic, Softmax, SVM đa lớp — cài đặt vectorized và **kiểm tra gradient** bằng sai phân hữu hạn.
- Huấn luyện bằng mini-batch SGD, chọn siêu tham số trên validation, đánh giá **một lần** trên test.
- Hiểu **giới hạn** của mô hình tuyến tính trên pixel thô — động lực cho mạng nơ-ron và CNN.

> Bài này hoàn thiện phần 2 của bài *One vs All* trong khoá AIVN (tuần 4): phân loại CIFAR-10 bằng các bộ phân loại tuyến tính của gói `algorithms/` (logistic, softmax, SVM, gradient check — dựa trên CS231n). [Bài 05](05_one_vs_all.ipynb) đã học One-vs-All trên `digits` (offline); ở đây làm trên dữ liệu thật như bản gốc và tự viết lại toàn bộ.
>
> ⏬ Lần chạy đầu tải ~170 MB (một lần, lưu trong `data/raw/` — xem [chương 03 · bài 01](../03_data_analysis_and_eda/01_types_of_data_and_storage.ipynb), mục 4).

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/07_logistic_regression

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np

from ml_course.real_datasets import CIFAR10_CLASSES, load_cifar10
from ml_course.visualization import plot_confusion_matrix, plot_learning_curve, set_style

set_style()
X_train_all, y_train_all, X_test_raw, y_test = load_cifar10()
print("train:", X_train_all.shape, X_train_all.dtype, "| test:", X_test_raw.shape)
print("số ảnh mỗi lớp (train):", np.bincount(y_train_all))

## 1. Dữ liệu CIFAR-10

Mỗi ảnh là mảng `(32, 32, 3)` số nguyên 0–255; 50 000 ảnh train và 10 000 ảnh test, **cân bằng** 10 lớp. Xem vài ảnh mỗi lớp:

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(6, 10, figsize=(12, 7.5))
for k, name in enumerate(CIFAR10_CLASSES):
    for row, i in enumerate(rng.choice(np.flatnonzero(y_train_all == k), 6, replace=False)):
        ax = axes[row, k]
        ax.imshow(X_train_all[i])
        ax.axis("off")
        if row == 0:
            ax.set_title(name, fontsize=10)
plt.suptitle("CIFAR-10: cùng một lớp nhưng tư thế, màu sắc, nền rất khác nhau")
plt.show()

Chia dữ liệu: **20 000** ảnh train (chọn ngẫu nhiên để bài chạy nhanh và nhẹ RAM), **2 000** ảnh validation (lấy từ phần train còn lại — dùng để chọn siêu tham số), **10 000** ảnh test chính thức (chỉ dùng một lần ở cuối).

## 2. Tiền xử lý

1. **Duỗi** ảnh thành vector $32 \cdot 32 \cdot 3 = 3072$ chiều, chia 255.
2. **Trừ ảnh trung bình** — tính trên **train**, áp dụng cho cả validation/test (quy tắc "fit trên train" của [chương 04](../04_data_preprocessing/02_feature_scaling.ipynb)).
3. **Bias trick**: thêm một cột toàn 1, khi đó $W$ có shape $(3073, 10)$ và hàng cuối chính là bias $\mathbf b$: $\mathbf s = W^\top \mathbf x$ thay cho $W^\top \mathbf x + \mathbf b$.

In [ ]:
idx = np.random.default_rng(42).permutation(len(y_train_all))           # cùng cách chia với chương 29 · bài 04
train_idx, val_idx = idx[:20_000], idx[20_000:22_000]


def flatten(images):
    return images.reshape(len(images), -1).astype(np.float32) / 255.0


X_tr, X_va, X_te = flatten(X_train_all[train_idx]), flatten(X_train_all[val_idx]), flatten(X_test_raw)
y_tr, y_va = y_train_all[train_idx], y_train_all[val_idx]
mean_image = X_tr.mean(axis=0)                                     # CHỈ tính trên train


def add_bias(X):
    return np.hstack([X - mean_image, np.ones((len(X), 1), np.float32)])


X_tr, X_va, X_te = add_bias(X_tr), add_bias(X_va), add_bias(X_te)
del X_train_all                                                     # giải phóng ~150 MB
print("X_tr", X_tr.shape, "| X_va", X_va.shape, "| X_te", X_te.shape)
plt.figure(figsize=(2.5, 2.5))
plt.imshow(mean_image.reshape(32, 32, 3))
plt.axis("off")
plt.title("ảnh trung bình (train)")
plt.show()

## 3. Một mô hình, ba hàm mất mát

Với ảnh $\mathbf x_i$, mô hình tính **điểm** của 10 lớp $\mathbf s_i = W^\top \mathbf x_i$ và dự đoán $\hat y_i = \arg\max_k s_{ik}$. Ba cách "phạt" các điểm số khác nhau ($y_i$ là lớp đúng):

| Mô hình | Loss của một mẫu | Gradient theo điểm $\partial L_i / \partial s_{ik}$ |
|---|---|---|
| **One-vs-All logistic** (bài 05) | $-\sum_k \big[t_{ik}\log\sigma(s_{ik}) + (1-t_{ik})\log(1-\sigma(s_{ik}))\big]$, $t_{ik} = [k = y_i]$ | $\sigma(s_{ik}) - t_{ik}$ |
| **Softmax** (bài 06) | $-\log \dfrac{e^{s_{iy_i}}}{\sum_j e^{s_{ij}}}$ | $p_{ik} - t_{ik}$ |
| **SVM đa lớp** (hinge) | $\sum_{j \ne y_i} \max(0,\ s_{ij} - s_{iy_i} + \Delta)$, $\Delta = 1$ | $[\text{lề } j \text{ bị vi phạm}]$ với $j \ne y_i$; $-\#\{\text{lề bị vi phạm}\}$ với $k = y_i$ |

Cộng thêm regularization $\frac{\lambda}{2}\lVert W\rVert^2$ (không phạt hàng bias). Gradient theo $W$: $\nabla_W L = X^\top G / n + \lambda W$ với $G$ là ma trận gradient theo điểm.

Cùng một vector điểm, hai loss "nghĩ" khác nhau. Ví dụ ảnh **mèo** với điểm (mèo 3.2, ô tô 5.1, ếch −1.7):

In [ ]:
s = np.array([3.2, 5.1, -1.7])
svm_loss = np.maximum(0, s[[1, 2]] - s[0] + 1).sum()
p = np.exp(s - s.max()) / np.exp(s - s.max()).sum()
print(f"SVM loss = {svm_loss:.2f} | softmax loss = {-np.log(p[0]):.2f} (P(mèo) = {p[0]:.3f})")
s_good = np.array([13.2, 5.1, -1.7])
p_good = np.exp(s_good - s_good.max()) / np.exp(s_good - s_good.max()).sum()
print(f"khi điểm mèo = 13.2: SVM loss = {np.maximum(0, s_good[[1, 2]] - s_good[0] + 1).sum():.2f} | "
      f"softmax loss = {-np.log(p_good[0]):.5f}")

SVM "hài lòng" ngay khi lớp đúng vượt các lớp khác **một lề** $\Delta$ — loss bằng 0 tuyệt đối. Softmax thì không bao giờ hoàn toàn hài lòng: loss luôn dương, luôn muốn đẩy xác suất lớp đúng lên gần 1 hơn. Cài đặt vectorized cả ba:

In [ ]:
def l2_penalty(W, reg):
    Wn = W[:-1]                                                     # không phạt bias (hàng cuối)
    grad = np.zeros_like(W)
    grad[:-1] = reg * Wn
    return 0.5 * reg * np.sum(Wn * Wn), grad


def ova_logistic_loss(W, X, y, reg):
    n = len(y)
    S = X @ W
    T = np.zeros_like(S)
    T[np.arange(n), y] = 1
    P = 1 / (1 + np.exp(-S))
    loss = -np.mean(np.sum(T * np.log(P + 1e-12) + (1 - T) * np.log(1 - P + 1e-12), axis=1))
    G = P - T
    r_loss, r_grad = l2_penalty(W, reg)
    return loss + r_loss, X.T @ G / n + r_grad


def softmax_loss(W, X, y, reg):
    n = len(y)
    S = X @ W
    S = S - S.max(axis=1, keepdims=True)                            # ổn định số học
    P = np.exp(S)
    P /= P.sum(axis=1, keepdims=True)
    loss = -np.mean(np.log(P[np.arange(n), y] + 1e-12))
    G = P.copy()
    G[np.arange(n), y] -= 1
    r_loss, r_grad = l2_penalty(W, reg)
    return loss + r_loss, X.T @ G / n + r_grad


def svm_loss(W, X, y, reg, delta=1.0):
    n = len(y)
    S = X @ W
    margins = np.maximum(0, S - S[np.arange(n), y][:, None] + delta)
    margins[np.arange(n), y] = 0
    loss = margins.sum() / n
    G = (margins > 0).astype(X.dtype)
    G[np.arange(n), y] = -G.sum(axis=1)
    r_loss, r_grad = l2_penalty(W, reg)
    return loss + r_loss, X.T @ G / n + r_grad


LOSSES = {"One-vs-All logistic": ova_logistic_loss, "Softmax": softmax_loss, "SVM đa lớp": svm_loss}

## 4. Kiểm tra gradient (gradient check)

Gradient viết tay rất dễ sai một dấu hay một chỉ số. Đối chiếu với **sai phân trung tâm** $\frac{f(w + h) - f(w - h)}{2h}$ tại vài phần tử ngẫu nhiên của $W$ — đây là việc mà file `gradient_check.py` trong gói gốc của AIVN làm. Dùng `float64` và một tập nhỏ để sai số làm tròn không che lấp kết quả.

In [ ]:
def grad_check_sparse(loss_fn, W, X, y, reg, n_checks=8, h=1e-5, seed=1):
    r = np.random.default_rng(seed)
    _, analytic = loss_fn(W, X, y, reg)
    errors = []
    for _ in range(n_checks):
        ix = tuple(r.integers(0, d) for d in W.shape)
        old = W[ix]
        W[ix] = old + h
        f_plus = loss_fn(W, X, y, reg)[0]
        W[ix] = old - h
        f_minus = loss_fn(W, X, y, reg)[0]
        W[ix] = old
        numeric = (f_plus - f_minus) / (2 * h)
        errors.append(abs(numeric - analytic[ix]) / max(1e-12, abs(numeric) + abs(analytic[ix])))
    return max(errors)


X_small, y_small = X_tr[:100].astype(np.float64), y_tr[:100]
W_check = 0.001 * np.random.default_rng(2).standard_normal((X_tr.shape[1], 10))
for name, loss_fn in LOSSES.items():
    print(f"{name:20s} sai số tương đối lớn nhất = {grad_check_sparse(loss_fn, W_check, X_small, y_small, reg=0.1):.2e}")

Sai số tương đối cỡ $10^{-7}$ trở xuống: gradient đúng. (Với SVM, thỉnh thoảng một phép thử rơi đúng "điểm gãy" của hàm $\max(0, \cdot)$ và cho sai số lớn hơn — đó không phải bug mà do hàm không khả vi tại điểm đó.)

## 5. Huấn luyện bằng mini-batch SGD và chọn siêu tham số

Mỗi bước lấy ngẫu nhiên 256 ảnh, tính loss và gradient, cập nhật $W \leftarrow W - \eta\nabla_W L$ ([chương 06 · bài 04](../06_linear_regression/04_batch_sgd_minibatch_gd.ipynb)). Chọn learning rate $\eta$ và $\lambda$ theo accuracy trên **validation** (lưới đủ rộng để giá trị tốt nhất không nằm ở biên — [chương 16](../16_hyperparameter_optimization/01_grid_vs_random_search.ipynb)):

In [ ]:
def train_sgd(loss_fn, X, y, lr, reg, n_iters=1500, batch_size=256, seed=0):
    r = np.random.default_rng(seed)
    W = (0.001 * r.standard_normal((X.shape[1], 10))).astype(np.float32)
    history = []
    for _ in range(n_iters):
        b = r.integers(0, len(y), batch_size)
        loss, grad = loss_fn(W, X[b], y[b], reg)
        W -= lr * grad.astype(np.float32)
        history.append(loss)
    return W, history


def accuracy(W, X, y):
    return float(np.mean(np.argmax(X @ W, axis=1) == y))


t0 = time.perf_counter()
results, best = [], {}
for name, loss_fn in LOSSES.items():
    for lr in [0.003, 0.01, 0.03, 0.1]:
        for reg in [0.001, 0.1]:
            W, history = train_sgd(loss_fn, X_tr, y_tr, lr, reg)
            results.append((name, lr, reg, accuracy(W, X_tr, y_tr), accuracy(W, X_va, y_va)))
            if name not in best or results[-1][4] > best[name]["val"]:
                best[name] = {"lr": lr, "reg": reg, "val": results[-1][4], "W": W, "history": history}
print(f"{'mô hình':20s} {'lr':>6s} {'λ':>6s} {'train':>7s} {'val':>7s}")
for name, lr, reg, acc_tr, acc_va in results:
    print(f"{name:20s} {lr:6.3f} {reg:6.3f} {acc_tr:7.3f} {acc_va:7.3f}")
print(f"({len(results)} lần huấn luyện, {time.perf_counter() - t0:.0f} giây)")

Giờ mới dùng tập **test** — một lần duy nhất, với cấu hình tốt nhất (theo validation) của mỗi mô hình:

In [ ]:
for name, b in best.items():
    b["test"] = accuracy(b["W"], X_te, y_test)
    print(f"{name:20s} lr = {b['lr']:<6} λ = {b['reg']:<6} | val = {b['val']:.3f} | TEST = {b['test']:.3f}")

fig, ax = plt.subplots(figsize=(8, 3.5))
hist = np.array(best["Softmax"]["history"])
ax.plot(hist, alpha=0.3, label="loss từng mini-batch")
ax.plot(np.convolve(hist, np.ones(50) / 50, mode="valid"), label="trung bình trượt 50 bước")
ax.set_xlabel("bước SGD")
ax.set_ylabel("softmax loss")
ax.legend()
ax.set_title("Softmax: loss trong quá trình huấn luyện")
plt.show()

**Nhận xét:** ba hàm mất mát cho accuracy test **gần như nhau** — khoảng 38–40%, gấp 4 lần đoán ngẫu nhiên (10%) nhưng còn rất xa con người. Khi đổi loss mà kết quả không đổi, giới hạn nằm ở **mô hình**, không phải ở loss. (Đây cũng là đính chính cho nhận xét của bản gốc rằng One-vs-All kém softmax nhiều: trên CIFAR-10, khoảng cách là nhỏ — xem thêm bài 05.)

Hai quan sát nữa từ bảng chọn siêu tham số:
- $\lambda$ nhỏ luôn tốt hơn: mô hình đang **thiếu** khả năng biểu diễn (high bias — mục 8), regularization chỉ làm nó yếu thêm.
- Các cấu hình đứng đầu chênh nhau dưới 1 điểm phần trăm — nhỏ hơn sai số thống kê của accuracy trên 2 000 ảnh validation ($\sqrt{0.4 \cdot 0.6 / 2000} \approx 1.1\%$). Đừng "tối ưu" quá kỹ những khác biệt nằm trong vùng nhiễu ([chương 15](../15_cross_validation/README.md)).

## 6. Mô hình đã học gì? Trọng số là "ảnh mẫu" của từng lớp

Điểm của lớp $k$ là tích vô hướng $\mathbf w_k^\top \mathbf x$ — mô hình so ảnh với **một ảnh mẫu** $\mathbf w_k$ cho mỗi lớp (template matching). Vẽ 3072 trọng số đầu của mỗi cột thành ảnh 32×32×3:

In [ ]:
W_best = best["Softmax"]["W"][:-1]                                  # bỏ hàng bias
fig, axes = plt.subplots(1, 10, figsize=(14, 2.2))
for k, ax in enumerate(axes):
    template = W_best[:, k].reshape(32, 32, 3)
    template = (template - template.min()) / (template.max() - template.min())
    ax.imshow(template)
    ax.set_title(CIFAR10_CLASSES[k], fontsize=10)
    ax.axis("off")
plt.suptitle("Trọng số softmax của từng lớp, vẽ thành ảnh")
plt.show()

Các "ảnh mẫu" là những vệt màu mờ: *máy bay* và *tàu thuỷ* có nền xanh (trời, biển), *ếch* và *hươu* thiên về xanh lá/nâu, *ô tô* là một khối đỏ ở giữa (nhiều ô tô trong dữ liệu màu đỏ). Một lớp chỉ có **một** ảnh mẫu, nên nó phải là "trung bình" của mọi tư thế và màu sắc — ô tô đỏ hay xanh, quay trái hay phải. Mô hình tuyến tính chủ yếu dựa vào **màu và nền**, không nhận ra **hình dạng**.

## 7. Nhầm lẫn giữa các lớp

In [ ]:
pred_te = np.argmax(X_te @ best["Softmax"]["W"], axis=1)
cm = np.zeros((10, 10), int)
np.add.at(cm, (y_test, pred_te), 1)
fig, ax = plt.subplots(figsize=(7.5, 6.5))
plot_confusion_matrix(cm, labels=CIFAR10_CLASSES, ax=ax, normalize=True, title="Softmax trên CIFAR-10 (test, chuẩn hoá theo hàng)")
plt.xticks(rotation=45, ha="right")
plt.show()
off = cm - np.diag(np.diag(cm))
for k in np.argsort(off.ravel())[::-1][:4]:
    i, j = divmod(k, 10)
    print(f"{CIFAR10_CLASSES[i]:>9s} bị đoán là {CIFAR10_CLASSES[j]:<9s}: {off[i, j]} ảnh")
print("accuracy theo lớp:", {c: round(float(cm[k, k] / cm[k].sum()), 2) for k, c in enumerate(CIFAR10_CLASSES)})

Các lỗi lớn nhất rơi vào những cặp **trông giống nhau về màu và bố cục** — những gì một ảnh mẫu tuyến tính nhìn thấy được.

## 8. Thêm dữ liệu có giúp không? — learning curve

Nếu mô hình bị **high bias**, thêm dữ liệu gần như không giúp ([chương 05 · bài 05](../05_machine_learning_fundamentals/05_learning_curves_and_error_analysis.ipynb)). Kiểm tra với softmax:

In [ ]:
sizes = [1000, 2500, 5000, 10_000, 20_000]
curve_tr, curve_va = [], []
for n in sizes:
    W, _ = train_sgd(softmax_loss, X_tr[:n], y_tr[:n], best["Softmax"]["lr"], best["Softmax"]["reg"], n_iters=2000)
    curve_tr.append(accuracy(W, X_tr[:n], y_tr[:n]))
    curve_va.append(accuracy(W, X_va, y_va))
fig, ax = plt.subplots(figsize=(7, 4))
plot_learning_curve(sizes, curve_tr, curve_va, ax=ax, metric="accuracy", title="Softmax trên CIFAR-10: hai đường hội tụ quanh ~40%")
plt.show()
for n, a_tr, a_va in zip(sizes, curve_tr, curve_va):
    print(f"n = {n:6,d}: train = {a_tr:.3f} | validation = {a_va:.3f}")

Với ít dữ liệu, mô hình **overfit** (train cao, validation thấp). Khi tăng dữ liệu, hai đường **hội tụ** về khoảng 40% — dấu hiệu kinh điển của **high bias**: thêm 30 000 ảnh còn lại hay 1 triệu ảnh nữa cũng không đưa mô hình tuyến tính trên pixel thô đi xa hơn bao nhiêu.

**Cần gì?** Một mô hình biểu diễn được **đặc trưng** (cạnh, góc, kết cấu, bộ phận vật thể) bất kể vị trí, màu sắc: mạng nơ-ron nhiều lớp ([chương 27](../27_deep_learning_foundations/README.md)) và đặc biệt **CNN** ([chương 29](../29_cnn/README.md)) — ở đó ta sẽ quay lại chính bộ CIFAR-10 này.

## 🧠 Kiểm tra nhanh

1. Ảnh có điểm (đúng = 2.0, hai lớp khác = 2.5 và 0.5). Tính SVM loss với $\Delta = 1$.
2. Vì sao ảnh trung bình phải tính trên **train** rồi áp dụng cho validation/test?
3. Khi lớp đúng đã vượt các lớp khác hơn $\Delta$, gradient của SVM loss theo $W$ (bỏ qua regularization) bằng bao nhiêu? Softmax thì sao?
4. Learning curve cho train ≈ validation ≈ 40%. Nên thu thêm dữ liệu hay đổi mô hình? Vì sao?
5. Vì sao "ảnh mẫu" tuyến tính của lớp *ngựa* có thể trông như con ngựa hai đầu?

## 📌 Tóm tắt

- CIFAR-10: 60 000 ảnh 32×32×3, 10 lớp cân bằng — chuẩn mực để so sánh mô hình thị giác.
- Tiền xử lý: duỗi + chia 255 + trừ ảnh trung bình (tính trên train) + bias trick.
- Một mô hình tuyến tính, ba loss (OvA logistic, softmax, SVM đa lớp) → accuracy test ~40% như nhau: **giới hạn là mô hình**.
- Luôn **kiểm tra gradient** bằng sai phân trung tâm trước khi tin vào gradient viết tay.
- Trọng số tuyến tính = một ảnh mẫu mỗi lớp → dựa vào màu/nền; learning curve hội tụ thấp → high bias → cần đặc trưng học được (CNN).

## 📚 Tài liệu tham khảo

- CS231n — [Linear classification](https://cs231n.github.io/linear-classify/), [Optimization](https://cs231n.github.io/optimization-1/) (Stanford).
- A. Krizhevsky (2009). *Learning Multiple Layers of Features from Tiny Images* — [trang CIFAR](https://www.cs.toronto.edu/~kriz/cifar.html).
- J. Weston, C. Watkins (1999). *Support Vector Machines for Multi-Class Pattern Recognition*.

<sub>Hoàn thiện phần 2 bài *Logistic Regression — One vs All* và gói `algorithms/` (linear classifier, loss/gradient logistic–softmax–SVM, `gradient_check`; dựa trên CS231n) của khoá [AIVN-Machine-Learning](https://github.com/thanhhff/AIVN-Machine-Learning), Tuần 4 (MIT License). Viết lại bằng NumPy; bổ sung so sánh ba loss, chọn siêu tham số trên validation, trực quan hoá trọng số, confusion matrix và learning curve.</sub>